# Test Model Accuracy on Age Categories (Colab)
This notebook runs on Google Colab. It mounts your Google Drive, pulls the latest code from the V2 repository, loads your trained model, and calculates the age category prediction accuracy on your validation dataset.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# 2. Clone or update the GitHub repo
if [ -d /content/AgeGenderClassifier/.git ]; then
  echo 'Repository already exists, pulling latest changes'
  cd /content/AgeGenderClassifier
  git pull
else
  git clone --depth 1 https://github.com/mohamedIzoughne/gender-age-prediction-mobileNetv3-modelV2.git /content/AgeGenderClassifier
  cd /content/AgeGenderClassifier
fi

# Install requirements
if [ -f requirements.txt ]; then
  pip install -r requirements.txt
else
  pip install torch torchvision pytorch_lightning pandas pillow
fi

In [ ]:
# 3. Set Working Directory
import sys, os
project_dir = '/content/AgeGenderClassifier'
if os.path.exists(project_dir):
    os.chdir(project_dir)
    if project_dir not in sys.path:
        sys.path.insert(0, project_dir)
    print('Working directory set to', project_dir)

In [ ]:
# 4. Define Age Categories (Custom definition, no external imports needed)
AGE_CLASSES = [
    {"id": 0, "label": "dependent_minor", "min_age": 0, "max_age": 17},
    {"id": 1, "label": "young_aspirational", "min_age": 18, "max_age": 24},
    {"id": 2, "label": "independent_professional", "min_age": 25, "max_age": 29},
    {"id": 3, "label": "early_nesting", "min_age": 30, "max_age": 35},
    {"id": 4, "label": "peak_earning_adult", "min_age": 36, "max_age": 45},
    {"id": 5, "label": "mature_provider", "min_age": 46, "max_age": 55},
    {"id": 6, "label": "pre_retirement", "min_age": 56, "max_age": 65},
    {"id": 7, "label": "senior_consumer", "min_age": 66, "max_age": 120},
]

def age_to_class(age: float) -> int:
    """Map continuous age to age class ID (0-7)."""
    age = int(age)
    for cls_info in AGE_CLASSES:
        if cls_info["min_age"] <= age <= cls_info["max_age"]:
            return cls_info["id"]
    return 7 if age > 65 else 0

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix, classification_report

# Import model loading function and datamodule from the cloned repo
from src.models.MobileNet.runner_scripts.trainer import load_model
from src.models.MobileNet.data_defs import AgeGenderDataModule

In [ ]:
# 5. Load the model
# Point this to the .pth file stored in your Google Drive
model_path = "/content/drive/MyDrive/AgeGenderCheckpoints/model_checkpoint.pth" # UPDATE IF DIFFERENT

model = load_model(model_path)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model loaded successfully!")

In [ ]:
# 6. Setup the DataModule to load the validation set
config = model.config

# By only providing 'ds_path' (and NOT train_path or val_path),
# the DataModule will automatically re-run the exact same 80/20 on-the-fly split
# using torch.manual_seed(42) that it used during training!
config["ds_path"] = "/content/drive/MyDrive/Datasets/UTKFace" # UPDATE THIS PATH

if "train_path" in config:
    del config["train_path"]
if "val_path" in config:
    del config["val_path"]

config["batch_size"] = 64

datamodule = AgeGenderDataModule(config, mode="fit")
datamodule.setup(stage="fit")

val_loader = datamodule.val_dataloader()
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# 7. Run Inference on the Validation Set
all_true_ages = []
all_pred_ages = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Predicting Validation Set"):
        x, true_age, true_gender, is_augmented, image_paths = batch
        x = x.to(device)
        
        _, age_pred = model(x)
        
        all_true_ages.extend(true_age.numpy())
        all_pred_ages.extend(age_pred.cpu().numpy())

In [ ]:
# 8. Convert Continuous Ages to Categories and Calculate Accuracy
y_true_cats = [age_to_class(age) for age in all_true_ages]
y_pred_cats = [age_to_class(age) for age in all_pred_ages]

correct = sum(1 for true, pred in zip(y_true_cats, y_pred_cats) if true == pred)
total = len(y_true_cats)
accuracy = (correct / total) * 100 if total > 0 else 0

print(f"Total Validation Samples: {total}")
print(f"Correct Category Predictions: {correct}")
print(f"Overall Category Accuracy: {accuracy:.2f}%")

In [ ]:
# 9. Plot Confusion Matrix
labels = [c["label"] for c in AGE_CLASSES]

cm = confusion_matrix(y_true_cats, y_pred_cats)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted Category')
plt.ylabel('True Category')
plt.title('Age Category Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 10. Detailed Classification Report
# This shows precision, recall, and f1-score per category
try:
    print(classification_report(y_true_cats, y_pred_cats, target_names=labels))
except Exception as e:
    print("Could not generate report (some classes might not be present in validation set).")
    print(e)